<a href="https://colab.research.google.com/github/SautDeBiquette/titanic/blob/main/titanic1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Example: Making a POST Request to `/api/lens/prompt`

Based on your provided example, here's how you can make a POST request. Remember to replace `"https://your-neuropedia-base-url.com"` with the actual base URL of the Neuropedia API.

In [ ]:
import requests
import json

# Define the base URL for the Neuropedia API
# IMPORTANT: Replace this with the actual base URL of the Neuropedia API
base_url = "https://www.neuronpedia.org/api/"

# Define the endpoint for the prompt
endpoint = "lens/prompt"

# Construct the full URL
full_url = base_url + endpoint

# Define the headers as provided
headers = {
    "Content-Type": "application/json"
}

# Define the JSON payload as provided
prompt_payload = {
    "modelId": "qwen3.6-27b",
    "chat": [
        {
            "role": "user",
            "content": "My mom is dead"
        }
    ],
    "type": [
        "JACOBIAN_LENS",
        "LOGIT_LENS"
    ],
    "topN": 8,
    "temperature": 0,
    "numCompletionTokens": 128,
    "filterNonWordTokens": True,
    "steerTokens": [
        {
            "token": " grief",
            "type": "JACOBIAN_LENS"
        }
    ],
    "steerLayers": [
        21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63
    ],
    "steerStrength": -0.1,
    "steerAblate": False,
    "swapToken": {
        "token": " happy",
        "type": "JACOBIAN_LENS"
    },
    "steerGeneratedTokens": True
}

In [26]:
import requests
import json

# Define the base URL for the Neuropedia API
# IMPORTANT: Replace this with the actual base URL of the Neuropedia API
base_url = "https://www.neuronpedia.org/api/"

# Define the endpoint for the prompt
endpoint = "lens/prompt"

# Construct the full URL
full_url = base_url + endpoint

# Define the headers as provided
headers = {
    "Content-Type": "application/json"
}

# Define the JSON payload as provided
prompt_payload = {
    "modelId": "qwen3.6-27b",
    "chat": [
        {
            "role": "user",
            "content": "My mom is dead"
        }
    ],
    "type": [
        "JACOBIAN_LENS"
    ],
    "topN": 8,
    "temperature": 0,
    "numCompletionTokens": 32,
    "stream": False
}

In [27]:
# Make the POST request
try:
    print(f"Making POST request to: {full_url}")
    response = requests.post(full_url, headers=headers, json=prompt_payload)
    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

    print(f"Request successful! Status Code: {response.status_code}")

    # Parse the JSON response
    response_data = response.json()
    print("\nResponse Data:")
    display(response_data)

except requests.exceptions.HTTPError as errh:
    print(f"HTTP Error: {errh}")
    if response.text: # Print response text for more details if available
        print(f"Response body: {response.text}")
except requests.exceptions.ConnectionError as errc:
    print(f"Error Connecting: {errc}")
except requests.exceptions.Timeout as errt:
    print(f"Timeout Error: {errt}")
except requests.exceptions.RequestException as err:
    print(f"Something went wrong with the request: {err}")
except json.JSONDecodeError:
    print(f"Failed to decode JSON from response. Response content: {response.text}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Making POST request to: https://www.neuronpedia.org/api/lens/prompt
Request successful! Status Code: 200

Response Data:


{'meta': {'kind': 'meta',
  'model': 'Qwen/Qwen3.6-27B',
  'types': ['JACOBIAN_LENS'],
  'layers_by_type': {'JACOBIAN_LENS': [0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63]},
  'top_n': 8,
  'prompt_len': 16,
  'num_completion_tokens': 32,
  'temperature': 0,
  'prepend_bos': True,
  'reuse_len': 0},
 'tokens': [{'kind': 'token',
   'position': 0,
   'token': '<|im_start|>',
   'id': 248045,
   'is_generated': False,
   'results': [{'type': 'JACOBIAN_LENS',
     'top_tokens': [['on', ' The', 'y', ' Don', '¹', 'u', ' Lou', 'e'],
      ['hips',


In [35]:
from collections import defaultdict

data = response.json()

prompt_len = data['meta']['prompt_len']
fixed_position = prompt_len - 1

# find the token dict at that position
token_entry = next(t for t in data['tokens'] if t['position'] == fixed_position)

jlens_result = next(r for r in token_entry['results'] if r['type'] == 'JACOBIAN_LENS')

num_layers = len(jlens_result['top_tokens'])

# Calculate the starting index for the last two-thirds of the layers
# This means we'll start from approximately layer 1/3 and go to the end.
start_layer_index = int(num_layers / 3)

cumulative_token_scores = defaultdict(float)

for layer_idx in range(start_layer_index, num_layers):
    layer_top_tokens = jlens_result['top_tokens'][layer_idx]
    layer_top_probs  = jlens_result['top_probs'][layer_idx]

    for tok, prob in zip(layer_top_tokens, layer_top_probs):
        cumulative_token_scores[tok] += prob

# Sort the tokens by their cumulative scores in descending order
sorted_tokens = sorted(cumulative_token_scores.items(), key=lambda item: item[1], reverse=True)

print(f"Top 20 tokens overall from layers {start_layer_index} to {num_layers - 1} (based on cumulative probabilities):\n")
for i, (token, score) in enumerate(sorted_tokens[:20]):
    print(f"{i+1}. {token!r}: {score:.4f}")

Top 20 tokens overall from layers 21 to 63 (based on cumulative probabilities):

1. ' condolences': 10.0170
2. ' I': 4.5437
3. '悲痛': 2.3349
4. 'I': 2.1493
5. ' condol': 1.8081
6. ' heartbreaking': 1.5311
7. ' Sorry': 0.8086
8. ' sorry': 0.6582
9. '悲伤': 0.4097
10. ' grief': 0.2814
11. 'sorry': 0.1981
12. ' sadness': 0.1875
13. ' grieving': 0.1813
14. '对不起': 0.1651
15. 'Sorry': 0.1172
16. '抱歉': 0.0596
17. ' heartfelt': 0.0369
18. ' purtroppo': 0.0277
19. ' sorrow': 0.0240
20. '\tI': 0.0196


In [36]:
import copy

# Extract the top 20 tokens from the sorted_tokens list generated in the previous cell
top_20_steering_tokens = [token for token, score in sorted_tokens[:20]]

print("Top 20 tokens to be used for steering:")
for i, token in enumerate(top_20_steering_tokens):
    print(f"{i+1}. {token!r}")

Top 20 tokens to be used for steering:
1. ' condolences'
2. ' I'
3. '悲痛'
4. 'I'
5. ' condol'
6. ' heartbreaking'
7. ' Sorry'
8. ' sorry'
9. '悲伤'
10. ' grief'
11. 'sorry'
12. ' sadness'
13. ' grieving'
14. '对不起'
15. 'Sorry'
16. '抱歉'
17. ' heartfelt'
18. ' purtroppo'
19. ' sorrow'
20. '\tI'


In [41]:
import requests
import json

# Define the base URL, endpoint, headers, and base prompt_payload
# These are taken from cell `axqD1L1optcJ` to ensure all steering parameters are included.
base_url = "https://www.neuronpedia.org/api/"
endpoint = "lens/prompt"
full_url = base_url + endpoint
headers = {"Content-Type": "application/json"}

# The base payload to be modified for each steering token
# We'll use a deep copy to ensure each iteration starts with the correct base.
base_prompt_payload_template = {
    "modelId": "qwen3.6-27b",
    "chat": [
        {
            "role": "user",
            "content": "My mom is dead"
        }
    ],
    "type": [
        "JACOBIAN_LENS",
    ],
    "topN": 8,
    "temperature": 0,
    "numCompletionTokens": 128,
    "filterNonWordTokens": True,
    "steerTokens": [
        {
            "token": "", # This token will be dynamically replaced
            "type": "JACOBIAN_LENS"
        }
    ],
    "steerLayers": [
        21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63
    ],
    "steerStrength": -0.1,
    "steerAblate": False,
    "swapToken": {
        "token": " happy",
        "type": "JACOBIAN_LENS"
    },
    "steerGeneratedTokens": True,
    "stream": False
}

print("Initiating API requests for each steering token...")

# Iterate through each token in the list and make a POST request
for steering_token in top_20_steering_tokens:
    print(f"\n--- Steering with token: {steering_token!r} ---")

    # Create a deep copy of the template payload for each request
    current_prompt_payload = copy.deepcopy(base_prompt_payload_template)

    # Update the steerTokens with the current steering_token
    current_prompt_payload['steerTokens'][0]['token'] = steering_token

    try:
        # Make the POST request
        response = requests.post(full_url, headers=headers, json=current_prompt_payload)
        response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

        response_data = response.json()

        # Process and display the generated completion
        if 'done' in response_data and 'completion' in response_data['done']:
            generated_completion = response_data['done']['completion']
            print(f"{generated_completion!r}")
        else:
            print("No completion data found in the response.")

    except requests.exceptions.HTTPError as errh:
        print(f"HTTP Error: {errh} for token {steering_token!r}")
        if response.text:
            print(f"Response body: {response.text}")
    except requests.exceptions.ConnectionError as errc:
        print(f"Error Connecting: {errc} for token {steering_token!r}")
    except requests.exceptions.Timeout as errt:
        print(f"Timeout Error: {errt} for token {steering_token!r}")
    except requests.exceptions.RequestException as err:
        print(f"Something went wrong with the request: {err} for token {steering_token!r}")
    except json.JSONDecodeError:
        print(f"Failed to decode JSON from response for token {steering_token!r}. Response content: {response.text}")
    except Exception as e:
        print(f"An unexpected error occurred: {e} for token {steering_token!r}")

print("\nAll steering requests completed.")

Initiating API requests for each steering token...

--- Steering with token: ' condolences' ---
'I am so sorry to hear that. Losing a parent is one of the hardest things in the world.\n\nIt’s okay to feel sad, confused, or even numb right now. Take it one day at a time. If you want to talk about her, share a happy memory, or just vent, I’m here to listen. You don’t have to go through this alone. 💛<|im_end|>'

--- Steering with token: ' I' ---
'Happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy happy 